In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 39.5 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers faiss-cpu librosa -q

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import faiss
import os
import pickle
import torch
import librosa
from transformers import ClapModel, ClapProcessor
from pathlib import Path

DRIVE_DIR = '/content/drive/MyDrive/mood2music'
AUDIO_DIR = '/content/fma_needed'
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
print(os.listdir('/content/drive/MyDrive/mood2music'))

['catalog.parquet', 'fma_needed.zip', 'clap_checkpoint.pkl', 'track_index_clap.faiss', 'track_id_map_clap.pkl']


In [ ]:
!unzip -q /content/drive/MyDrive/mood2music/fma_needed.zip -d /content/fma_needed
print("Extraction done")
print(f"Files extracted: {len(list(Path('/content/fma_needed').rglob('*.mp3')))}")

replace /content/fma_needed/000/000002.mp3? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
Extraction done
Files extracted: 4816


In [ ]:
catalog = duckdb.query(f"""
    SELECT track_id, title, artist_name, genre_top,
           valence, energy, danceability, acousticness,
           instrumentalness, tempo_norm, speechiness,
           valence_reliable
    FROM '{DRIVE_DIR}/catalog.parquet'
    WHERE valence_reliable = true
""").df()

print(f"Catalog loaded: {len(catalog)} tracks")

Catalog loaded: 11868 tracks


In [ ]:
TARGET_SR = 48000

def get_audio_path(track_id, audio_dir):
    tid_str = str(track_id).zfill(6)
    subdir = tid_str[:3]
    return os.path.join(audio_dir, subdir, f"{tid_str}.mp3")

def load_audio(path, target_sr=TARGET_SR):
    """
    Librosa-only loading — stable across all platforms.
    torchaudio excluded due to torchcodec instability on macOS 26 beta.
    """
    try:
        audio, _ = librosa.load(path, sr=target_sr, mono=True)
        return audio
    except Exception as e:
        return None

In [ ]:
def load_clap_model():
    if torch.cuda.is_available():
        device = 'cuda'
    elif torch.backends.mps.is_available():
        device = 'mps'
    else:
        device = 'cpu'

    print(f"Device: {device}")
    model = ClapModel.from_pretrained("laion/clap-htsat-fused")
    processor = ClapProcessor.from_pretrained("laion/clap-htsat-fused")
    model = model.to(device)
    model.eval()
    print("Model loaded: laion/clap-htsat-fused")
    return model, processor, device

In [ ]:
# Run this after unzipping audio files
def find_available_tracks(catalog, audio_dir):
    available = []
    missing = []
    for _, row in catalog.iterrows():
        path = get_audio_path(row['track_id'], audio_dir)
        if os.path.exists(path):
            available.append(row['track_id'])
        else:
            missing.append(row['track_id'])
    print(f"Tracks with audio:    {len(available)}")
    print(f"Tracks without audio: {len(missing)}")
    return catalog[catalog['track_id'].isin(available)].reset_index(drop=True)

catalog_audio = find_available_tracks(catalog, AUDIO_DIR)

Tracks with audio:    4816
Tracks without audio: 7052


In [ ]:
def embed_audio_catalog(catalog_audio, model, processor, device, batch_size=32):
    """
    Batch size 32 is safe on A100 (40GB GPU memory).
    At 4816 tracks / 32 per batch = 151 batches.
    Estimated time: 10-15 minutes on A100.
    """
    embeddings = []
    valid_ids = []
    failed_ids = []

    tracks = catalog_audio.to_dict('records')
    total_batches = len(tracks) // batch_size + 1

    print(f"Embedding {len(tracks)} tracks in batches of {batch_size}")
    print(f"Estimated batches: {total_batches}\n")

    for batch_start in range(0, len(tracks), batch_size):
        batch = tracks[batch_start:batch_start + batch_size]
        batch_num = batch_start // batch_size + 1

        audio_batch = []
        batch_ids = []

        for track in batch:
            path = get_audio_path(track['track_id'], AUDIO_DIR)
            audio = load_audio(path)
            if audio is not None:
                audio_batch.append(audio)
                batch_ids.append(track['track_id'])
            else:
                failed_ids.append(track['track_id'])

        if not audio_batch:
            continue

        try:
            with torch.no_grad():
                inputs = processor(
                    audios=audio_batch,
                    sampling_rate=TARGET_SR,
                    return_tensors="pt",
                    padding=True
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}
                audio_embeds = model.get_audio_features(**inputs)
                audio_embeds = audio_embeds.cpu().numpy()

            embeddings.append(audio_embeds)
            valid_ids.extend(batch_ids)

        except Exception as e:
            print(f"Batch {batch_num} failed: {e}")
            failed_ids.extend(batch_ids)
            continue

        if batch_num % 10 == 0:
            print(f"Batch {batch_num}/{total_batches} — "
                  f"{len(valid_ids)} embedded, "
                  f"{len(failed_ids)} failed")

        if batch_num % 30 == 0:
            checkpoint = {
                'embeddings': np.vstack(embeddings),
                'valid_ids': valid_ids,
                'failed_ids': failed_ids,
                'last_batch': batch_num
            }
            with open(os.path.join(DRIVE_DIR, 'clap_checkpoint.pkl'), 'wb') as f:
                pickle.dump(checkpoint, f)
            print(f"  → checkpoint saved at batch {batch_num}")

    print(f"\nDone. Embedded: {len(valid_ids)}, Failed: {len(failed_ids)}")
    return np.vstack(embeddings), valid_ids, failed_ids

In [ ]:
model, processor, device = load_clap_model()

Device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model loaded: laion/clap-htsat-fused


In [ ]:
embeddings, valid_ids, failed_ids = embed_audio_catalog(
    catalog_audio, model, processor, device, batch_size=32
)

Embedding 4816 tracks in batches of 32
Estimated batches: 151

Batch 10/151 — 320 embedded, 0 failed
Batch 20/151 — 640 embedded, 0 failed
Batch 30/151 — 960 embedded, 0 failed
  → checkpoint saved at batch 30
Batch 40/151 — 1280 embedded, 0 failed
Batch 50/151 — 1600 embedded, 0 failed
Batch 60/151 — 1920 embedded, 0 failed
  → checkpoint saved at batch 60
Batch 70/151 — 2240 embedded, 0 failed
Batch 80/151 — 2560 embedded, 0 failed
Batch 90/151 — 2880 embedded, 0 failed
  → checkpoint saved at batch 90
Batch 100/151 — 3200 embedded, 0 failed
Batch 110/151 — 3520 embedded, 0 failed
Batch 120/151 — 3840 embedded, 0 failed
  → checkpoint saved at batch 120
Batch 130/151 — 4160 embedded, 0 failed
Batch 140/151 — 4480 embedded, 0 failed
Batch 150/151 — 4800 embedded, 0 failed
  → checkpoint saved at batch 150

Done. Embedded: 4816, Failed: 0


In [ ]:
def build_and_save_index(embeddings, valid_ids):
    dimension = embeddings.shape[1]
    print(f"Building index: {len(valid_ids)} vectors, {dimension} dimensions")

    embeddings_norm = embeddings.copy()
    faiss.normalize_L2(embeddings_norm)

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings_norm)

    # Save to Drive so it persists after Colab session ends
    faiss.write_index(index, os.path.join(DRIVE_DIR, 'track_index_clap.faiss'))
    with open(os.path.join(DRIVE_DIR, 'track_id_map_clap.pkl'), 'wb') as f:
        pickle.dump(valid_ids, f)

    size = os.path.getsize(os.path.join(DRIVE_DIR, 'track_index_clap.faiss'))
    print(f"Index saved to Drive. Size: {size / 1024**2:.1f} MB")
    return index

In [ ]:
index = build_and_save_index(embeddings, valid_ids)

Building index: 4816 vectors, 512 dimensions
Index saved to Drive. Size: 9.4 MB


In [ ]:
def embed_text_query(query_text, model, processor, device):
    with torch.no_grad():
        inputs = processor(text=[query_text], return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        text_embeds = model.get_text_features(**inputs)
        return text_embeds.cpu().numpy()

In [ ]:
def search_tracks_clap(query_text, index, track_id_map, catalog,
                        model, processor, device, k=5):
    query_vector = embed_text_query(query_text, model, processor, device)
    faiss.normalize_L2(query_vector)

    scores, positions = index.search(query_vector, k)

    results = []
    for score, pos in zip(scores[0], positions[0]):
        track_id = track_id_map[pos]
        track = catalog[catalog['track_id'] == track_id].iloc[0]
        results.append({
            'title': track['title'],
            'artist': track['artist_name'],
            'genre': track['genre_top'],
            'valence': round(track['valence'], 3),
            'energy': round(track['energy'], 3),
            'similarity': round(float(score), 3)
        })

    return results

In [ ]:
test_queries = [
    "I feel melancholy and tired, want something that matches my mood",
    "I need high energy music to get through a workout",
    "Something calm and peaceful for late night reading",
    "I'm feeling nostalgic and bittersweet"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    results = search_tracks_clap(query, index, valid_ids, catalog_audio,
                                  model, processor, device, k=3)
    for r in results:
        print(f"  {r['title']} — {r['artist']} ({r['genre']})")
        print(f"  valence={r['valence']}, energy={r['energy']}, similarity={r['similarity']}")


Query: 'I feel melancholy and tired, want something that matches my mood'
------------------------------------------------------------
  A Life In A Day — Candlestickmaker (Electronic)
  valence=0.054, energy=0.731, similarity=0.395
  Оля Зимой — Dragan Espenschied (Electronic)
  valence=0.783, energy=0.376, similarity=0.358
  Don't Think Twice (reprise) — et_ (Electronic)
  valence=0.164, energy=0.008, similarity=0.352

Query: 'I need high energy music to get through a workout'
------------------------------------------------------------
  The Simple Life — The Model (Electronic)
  valence=0.684, energy=0.573, similarity=0.566
  End Of Days — Light Asylum (Electronic)
  valence=0.737, energy=0.867, similarity=0.564
  Malta — Superhumanoids (Pop)
  valence=0.623, energy=0.922, similarity=0.539

Query: 'Something calm and peaceful for late night reading'
------------------------------------------------------------
  Suicide — Dani Shivers (Electronic)
  valence=0.28, energy=0.876, simi